# Week 44: Paired challenge evaluation

In [61]:
%pip install -q evaluate

In [62]:
import json
import re
import sys
from pathlib import Path

import evaluate
import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import transformers
from datasets import Dataset
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support, f1_score
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments
from transformers import set_seed

set_seed(42)

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("CUDA:", torch.version.cuda)

Python: 3.13.18
scikit-learn: 1.6.1
NumPy: 2.1.3
SciPy: 1.16.3
PyTorch: 2.11.0+cpu
Transformers: 5.20.2
GPU: None
CUDA: None


In [63]:
records = json.loads(Path("challenge.json").read_text(encoding="utf-8"))
challenge = pd.DataFrame(records)

# Selected on validation, never on the challenge set.
NO_ANSWER_THRESHOLD = -0.6354904174804688
MAX_ANSWER_LENGTH = 30

answerability_predictions = {}
answer_predictions = {}
sgd_token_predictions = {}
sgd_start_predictions = {}

## Week 36 rule and Week 39 logistic regression

In [64]:
def extract_numbers(text):
    return {''.join(str(int(digit)) for digit in number)
            for number in re.findall(r'\d+', text)}

def rule_answerable(question, context):
    question_numbers = extract_numbers(question)
    context_numbers = extract_numbers(context)
    return not question_numbers or bool(question_numbers & context_numbers)

answerability_predictions["Number-overlap rule"] = [
    rule_answerable(row["question"], row["context"]) for row in records
]

saved = joblib.load("week39-logistic-regression/model.joblib")
texts = challenge["question"] + " " + challenge["context"]
features = saved["vectorizer"].transform(texts)
answerability_predictions["Logistic regression"] = saved["model"].predict(features)

## Week 37 sequence labeller

In [65]:
tokenizer = AutoTokenizer.from_pretrained("multilingual-qa", use_fast=True, local_files_only=True)

def tokenize_with_offsets(text):
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
    return encoded.tokens(), encoded["offset_mapping"]

def token_features(tokens, index, question_tokens):
    word = tokens[index]
    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
    }
    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })
    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })
    for token in question_tokens:
        features["question." + token.lower()] = True
    return features

def valid_bio_transition(previous, current):
    if not current.startswith("I-"):
        return True
    if previous is None:
        return False
    entity_type = current[2:]
    return previous in {f"B-{entity_type}", f"I-{entity_type}"}

def constrained_beam_decode(token_log_probs, classes, beam_size=4):
    beam = [([], 0.0)]
    for scores in token_log_probs:
        candidates = []
        for sequence, sequence_score in beam:
            previous = sequence[-1] if sequence else None
            for column, label in enumerate(classes):
                if label == "B-ANS" and "B-ANS" in sequence:
                    continue
                if valid_bio_transition(previous, label):
                    candidates.append((sequence + [label], sequence_score + float(scores[column])))
        candidates.sort(key=lambda item: item[1], reverse=True)
        beam = candidates[:beam_size]
    return beam[0][0]

examples = []
for row in records:
    tokens, offsets = tokenize_with_offsets(row["context"])
    question_tokens, _ = tokenize_with_offsets(row["question"])
    gold_labels = ["O"] * len(tokens)
    if row["answerable"]:
        answer_end = row["answer_start"] + len(row["answer"])
        covered = [i for i, (start, end) in enumerate(offsets)
                   if start < answer_end and end > row["answer_start"]]
        assert covered
        gold_labels[covered[0]] = "B-ANS"
        for i in covered[1:]:
            gold_labels[i] = "I-ANS"
    examples.append({
        "offsets": offsets,
        "labels": gold_labels,
        "features": [token_features(tokens, i, question_tokens) for i in range(len(tokens))],
    })

In [66]:
for seed in [42, 43]:
    saved = joblib.load(f"week37-sgd/seed{seed}.joblib")
    model = saved["model"]
    vectorizer = saved["vectorizer"]
    name = f"SGD seed {seed}"
    predicted_answers = []
    predicted_starts = []
    predicted_labels = []
    for row, example in zip(records, examples):
        features = vectorizer.transform(example["features"]).astype(np.int32)
        scores = model.predict_log_proba(features)
        labels = constrained_beam_decode(scores, model.classes_, beam_size=4)
        predicted_labels.append(labels)
        answer_indices = [i for i, label in enumerate(labels) if label != "O"]
        start, answer = -1, ""
        if answer_indices:
            start = example["offsets"][answer_indices[0]][0]
            end = example["offsets"][answer_indices[-1]][1]
            answer = row["context"][start:end]
        predicted_starts.append(start)
        predicted_answers.append(answer)
    answer_predictions[name] = predicted_answers
    sgd_start_predictions[name] = predicted_starts
    sgd_token_predictions[name] = predicted_labels
    answerability_predictions[name] = [answer != "" for answer in predicted_answers]

## Integrated multilingual QA

In [67]:
model = AutoModelForQuestionAnswering.from_pretrained("multilingual-qa", local_files_only=True)
training_args = TrainingArguments(
    output_dir="week44-results",
    per_device_eval_batch_size=8,
    fp16=True,
    seed=42,
    full_determinism=True,
    report_to="none",
)
trainer = Trainer(model=model, args=training_args)

def prepare_challenge(batch):
    tokens = tokenizer(
        batch["question"], batch["context"],
        truncation="only_second", max_length=384, stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    tokens["example_index"] = tokens.pop("overflow_to_sample_mapping")
    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]
    return tokens

challenge_dataset = Dataset.from_list(records)
qa_features = challenge_dataset.map(
    prepare_challenge, batched=True, batch_size=len(records),
    remove_columns=challenge_dataset.column_names,
)
model_inputs = qa_features.remove_columns(["offset_mapping", "example_index"])
start_logits, end_logits = trainer.predict(model_inputs).predictions

null_scores = np.full(len(records), np.inf)
span_scores = np.full(len(records), -np.inf)
best_answers = [""] * len(records)
for feature_index, feature in enumerate(qa_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]
    cls_index = feature["input_ids"].index(tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index], float(starts[cls_index] + ends[cls_index])
    )
    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue
        for end in range(start, min(start + MAX_ANSWER_LENGTH, len(offsets))):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue
            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                best_answers[example_index] = records[example_index]["context"][start_offset[0]:end_offset[1]]

assert np.isfinite(null_scores).all()
assert np.isfinite(span_scores).all()
no_answer_scores = null_scores - span_scores
answer_predictions["Integrated QA"] = [
    answer if score <= NO_ANSWER_THRESHOLD else ""
    for answer, score in zip(best_answers, no_answer_scores)
]
answerability_predictions["Integrated QA"] = [
    answer != "" for answer in answer_predictions["Integrated QA"]
]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Answerability metrics

In [68]:
answerability_rows = []
for language in ["all", "ar", "ko", "te"]:
    mask = np.ones(len(challenge), dtype=bool)
    if language != "all":
        mask = challenge["lang"] == language
    gold = challenge.loc[mask, "answerable"]
    for name, predictions in answerability_predictions.items():
        precision, recall, f1, _ = precision_recall_fscore_support(
            gold, np.asarray(predictions)[mask],
            labels=[False, True], zero_division=0,
        )
        answerability_rows.append({
            "model": name,
            "language": language,
            "n": len(gold),
            "unanswerable_P": precision[0],
            "unanswerable_R": recall[0],
            "unanswerable_F1": f1[0],
            "answerable_P": precision[1],
            "answerable_R": recall[1],
            "answerable_F1": f1[1],
            "macro_F1": f1.mean(),
        })

answerability_results = pd.DataFrame(answerability_rows)
display(answerability_results.round(4))

,model,language,n,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,macro_F1
0,Number-overlap rule,all,30,0.0000,0.0000,0.0000,0.5000,1.0000,0.6667,0.3333
1,Logistic regression,all,30,0.0000,0.0000,0.0000,0.5000,1.0000,0.6667,0.3333
2,SGD seed 42,all,30,0.4444,0.2667,0.3333,0.4762,0.6667,0.5556,0.4444
3,SGD seed 43,all,30,0.4444,0.2667,0.3333,0.4762,0.6667,0.5556,0.4444
4,Integrated QA,all,30,1.0000,0.1333,0.2353,0.5357,1.0000,0.6977,0.4665
5,Number-overlap rule,ar,10,0.0000,0.0000,0.0000,0.5000,1.0000,0.6667,0.3333
6,Logistic regression,ar,10,0.0000,0.0000,0.0000,0.5000,1.0000,0.6667,0.3333
7,SGD seed 42,ar,10,0.5000,0.2000,0.2857,0.5000,0.8000,0.6154,0.4505
8,SGD seed 43,ar,10,0.5000,0.2000,0.2857,0.5000,0.8000,0.6154,0.4505
9,Integrated QA,ar,10,0.0000,0.0000,0.0000,0.5000,1.0000,0.6667,0.3333


## SGD span metrics

In [69]:
sgd_rows = []
for language in ["all", "ar", "ko", "te"]:
    for name, predictions in sgd_token_predictions.items():
        gold_tokens = []
        predicted_tokens = []
        exact_matches = []
        for i, row in enumerate(records):
            if language != "all" and row["lang"] != language:
                continue
            gold_tokens.extend(label != "O" for label in examples[i]["labels"])
            predicted_tokens.extend(label != "O" for label in predictions[i])
            predicted_span = (sgd_start_predictions[name][i], answer_predictions[name][i])
            gold_span = (row["answer_start"], row["answer"])
            exact_matches.append(predicted_span == gold_span)
        f1 = f1_score(gold_tokens, predicted_tokens, zero_division=0)
        sgd_rows.append({
            "model": name,
            "language": language,
            "n": len(exact_matches),
            "token_F1": f1,
            "exact_span_match": np.mean(exact_matches),
        })

sgd_results = pd.DataFrame(sgd_rows)
display(sgd_results.round(4))

,model,language,n,token_F1,exact_span_match
0,SGD seed 42,all,30,0.1758,0.2000
1,SGD seed 43,all,30,0.1053,0.1333
2,SGD seed 42,ar,10,0.0588,0.1000
3,SGD seed 43,ar,10,0.0500,0.1000
4,SGD seed 42,ko,10,0.0000,0.2000
5,SGD seed 43,ko,10,0.0000,0.2000
6,SGD seed 42,te,10,0.3415,0.3000
7,SGD seed 43,te,10,0.2051,0.1000


## QA span metrics

In [70]:
squad_metric = evaluate.load("squad_v2")
qa_predictions = []
qa_gold_answers = []
for i, row in enumerate(records):
    prediction = {}
    prediction["id"] = row["id"]
    prediction["prediction_text"] = answer_predictions["Integrated QA"][i]
    # The frozen threshold has already determined whether to abstain.
    prediction["no_answer_probability"] = 0.0
    qa_predictions.append(prediction)

    answer = {}
    if row["answerable"]:
        answer["text"] = [row["answer"]]
        answer["answer_start"] = [row["answer_start"]]
    else:
        answer["text"] = []
        answer["answer_start"] = []

    gold_answer = {}
    gold_answer["id"] = row["id"]
    gold_answer["answers"] = answer
    qa_gold_answers.append(gold_answer)

In [71]:
qa_rows = []
for language in ["all", "ar", "ko", "te"]:
    predictions = []
    gold_answers = []
    for i, row in enumerate(records):
        if language != "all" and row["lang"] != language:
            continue
        predictions.append(qa_predictions[i])
        gold_answers.append(qa_gold_answers[i])
    scores = squad_metric.compute(predictions=predictions, references=gold_answers)
    qa_rows.append({
        "model": "Integrated QA",
        "language": language,
        "n": len(predictions),
        "F1": scores["f1"] / 100,
        "EM": scores["exact"] / 100,
    })

qa_results = pd.DataFrame(qa_rows)
display(qa_results.round(4))

,model,language,n,F1,EM
0,Integrated QA,all,30,0.4519,0.4
1,Integrated QA,ar,10,0.5000,0.5
2,Integrated QA,ko,10,0.4750,0.4
3,Integrated QA,te,10,0.3807,0.3


## Prediction correctness

In [72]:
correct_predictions = pd.DataFrame(index=challenge.index)

for name in ["Number-overlap rule", "Logistic regression"]:
    predictions = answerability_predictions[name]
    correct_predictions[name] = predictions == challenge["answerable"]

for name in sgd_start_predictions:
    correct = []
    for i, row in enumerate(records):
        predicted_start = sgd_start_predictions[name][i]
        predicted_answer = answer_predictions[name][i]
        predicted_span = (predicted_start, predicted_answer)
        gold_span = (row["answer_start"], row["answer"])
        correct.append(predicted_span == gold_span)
    correct_predictions[name] = correct

correct = []
for i in range(len(records)):
    prediction = qa_predictions[i]
    gold_answer = qa_gold_answers[i]
    score = squad_metric.compute(predictions=[prediction], references=[gold_answer])
    correct.append(score["exact"] == 100)
correct_predictions["Integrated QA"] = correct

## Paired results

In [73]:
for name in correct_predictions:
    correct = correct_predictions[name]

    correct_answerable = correct[challenge["answerable"]].sum()
    correct_unanswerable = correct[~challenge["answerable"]].sum()

    correct_pairs = 0
    total_pairs = 0
    for pair_id, pair in challenge.groupby("pair_id", sort=False):
        if correct[pair.index].all():
            correct_pairs += 1
        total_pairs += 1

    print("\nMODEL:", name)
    print("Correct answerable:", correct_answerable, "/", challenge["answerable"].sum())
    print("Correct unanswerable:", correct_unanswerable, "/", (~challenge["answerable"]).sum())
    print("Correct pairs:", correct_pairs, "/", total_pairs)


MODEL: Number-overlap rule
Correct answerable: 15 / 15
Correct unanswerable: 0 / 15
Correct pairs: 0 / 15

MODEL: Logistic regression
Correct answerable: 15 / 15
Correct unanswerable: 0 / 15
Correct pairs: 0 / 15

MODEL: SGD seed 42
Correct answerable: 2 / 15
Correct unanswerable: 4 / 15
Correct pairs: 0 / 15

MODEL: SGD seed 43
Correct answerable: 0 / 15
Correct unanswerable: 4 / 15
Correct pairs: 0 / 15

MODEL: Integrated QA
Correct answerable: 10 / 15
Correct unanswerable: 2 / 15
Correct pairs: 1 / 15


## Failed pairs

In [74]:
for name in correct_predictions:
    correct = correct_predictions[name]
    print("\nMODEL:", name)
    for pair_id, pair in challenge.groupby("pair_id", sort=False):
        both_correct = correct[pair.index].all()
        if both_correct:
            continue
        first_question = pair.iloc[0]
        print("\nPAIR:", pair_id)
        print("Language:", first_question["lang"])
        print("Context:", first_question["context"])
        for i, row in pair.iterrows():
            print("ID:", row["id"], "Correct:", bool(correct[i]))
            print("Question:", row["question"])
            print("Gold answerable:", row["answerable"])
            print("Gold answer:", repr(row["answer"]))
            print("Predicted answerable:", bool(answerability_predictions[name][i]))
            if name in answer_predictions:
                print("Predicted answer:", repr(answer_predictions[name][i]))


MODEL: Number-overlap rule

PAIR: bgv600_nordkorea
Language: ar
Context: North Korea, officially the Democratic People's Republic of Korea (DPRK), is a country in East Asia.
ID: bgv600_nordkorea_answerable Correct: True
Question: أين تقع كوريا الشمالية؟
Gold answerable: True
Gold answer: 'East Asia'
Predicted answerable: True
ID: bgv600_nordkorea_unanswerable Correct: False
Question: أين تقع الدنمارك؟
Gold answerable: False
Gold answer: ''
Predicted answerable: True

PAIR: bgv600_FrankieGaye
Language: ar
Context: Starting in the mid-1970s, Frankie began working with his brother, joining him on the road during Gaye's concert tours and sometimes, to test audience reactions, Marvin would place Frankie onstage first before he arrived.
ID: bgv600_FrankieGaye_answerable Correct: True
Question: متى بدأ فرانكي العمل مع شقيقه؟
Gold answerable: True
Gold answer: 'mid-1970s'
Predicted answerable: True
ID: bgv600_FrankieGaye_unanswerable Correct: False
Question: متى توقف فرانكي عن العمل مع شقيقه؟